# NewsBias Article
### Independent article-level political leaning

We independently fine-tune **Ettin-400M** on the project's pinned AllSides corpus. The output classes are **left**, **center**, and **right**. This model has its own encoder weights, separate from NewsBias Sentence.

**Architecture:** overlapping passages → shared passage encoder → masked passage vectors → lightweight document transformer → gated attention → residual article classifier.

**Session protocol:** upload the immutable `payload.zip` and previous `resume.zip`, run for a 15-minute budget, download the new checkpoint, and continue in a fresh runtime. Training is partitioned by optimizer progress, not independent data shards.

Development uses random example splits across available outlets. Final `full` training uses all eligible records with the previously selected epoch count; no outlet is held out.

In [ ]:
import os
import time
from pathlib import Path

SESSION_STARTED = float(os.environ.get('NEWSBIAS_SESSION_STARTED', time.monotonic()))
HEADLESS = os.environ.get('NEWSBIAS_HEADLESS') == '1'
ACTION = os.environ.get('NEWSBIAS_ACTION', 'train')
print(f'Article specialist · {ACTION} · 15-minute resumable session')

## 1. Runtime and uploaded inputs
All article records, source code, and initial language weights are prepared locally before dispatch. A previous resume archive restores your trained model and optimizer. For interactive use, upload or mount the prepared archives first.

In [ ]:
import json
import subprocess
import sys
import zipfile

RUNTIME_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
INPUT_ROOT = Path(os.environ.get('NEWSBIAS_INPUT_ROOT', RUNTIME_ROOT / 'newsbias-inputs'))
OUTPUT_ROOT = Path(os.environ.get('NEWSBIAS_OUTPUT_ROOT', RUNTIME_ROOT / 'outputs'))
if not (INPUT_ROOT / 'config.json').exists():
    candidates = [RUNTIME_ROOT / 'payload.zip', *Path('/kaggle/input').glob('**/payload.zip')]
    payload = next((path for path in candidates if path.is_file()), None)
    if payload is None:
        raise FileNotFoundError('Upload or mount the prepared payload.zip before running.')
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(payload) as archive:
        if any(not (INPUT_ROOT / item.filename).resolve().is_relative_to(INPUT_ROOT.resolve()) for item in archive.infolist()):
            raise ValueError('Invalid input archive path')
        archive.extractall(INPUT_ROOT)
if not HEADLESS:
    requirements = json.loads((INPUT_ROOT / 'requirements.json').read_text())
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *requirements], check=True)
sys.path.insert(0, str(INPUT_ROOT))

import pandas as pd
import torch
from scripts.training import SessionTrainer, TrainingConfig, plot_history

if HEADLESS and not torch.cuda.is_available():
    raise RuntimeError('The headless training worker requires the allocated CUDA GPU.')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyTorch:', torch.__version__)

## 2. Article data and frozen recipe
The source is `upasanachatterjee/AllSides-random-split` at revision `4d33d17c5591ad5d4e37ffd6238d7ead8590a70e`. Body text is the input; readable labels determine the fixed output IDs. Text deduplication and splitting happen before passage construction. The preparation script writes the exact record membership into the payload.

In [ ]:
from dataclasses import asdict

config = TrainingConfig(**json.loads((INPUT_ROOT / 'config.json').read_text()))
assert config.task == 'article', 'This notebook requires an article payload.'
prepared = json.loads((INPUT_ROOT / 'data.json').read_text())
display(pd.Series(asdict(config), name='Value').to_frame())
distribution = pd.DataFrame([
    {'split': split, 'label': label, 'count': sum(row['label'] == label for row in rows)}
    for split, rows in prepared['splits'].items() for label in config.labels
])
display(distribution.pivot(index='split', columns='label', values='count'))
del prepared

## 3. Hierarchical article model

- Encode 1,024-token passages with 128-token overlap; retain at most 16 using deterministic, evenly spaced coverage.
- Project each 1,024-wide masked mean vector to width 256 and add passage-position embeddings.
- Apply two pre-norm document transformer blocks with four attention heads and a 1,024-wide feed-forward layer.
- Pool valid passages with gated attention.
- Add a learned document correction to the log of mean passage probabilities. The final residual projection starts at zero.

One cross-entropy loss is applied per article. Article labels are not assigned as sentence-level supervision. All encoder layers, passage heads, and document heads receive gradients. Passage microbatching retains autograd; gradient checkpointing controls activation memory.

In [ ]:
resume_path = Path(os.environ.get('NEWSBIAS_RESUME', RUNTIME_ROOT / 'resume.zip'))
if not resume_path.exists():
    mounted = list(Path('/kaggle/input').glob('**/resume.zip'))
    resume_path = mounted[0] if len(mounted) == 1 else resume_path
trainer = SessionTrainer(
    config, INPUT_ROOT / 'data.json', OUTPUT_ROOT,
    resume_archive=resume_path if resume_path.exists() else None,
    initial_model=INPUT_ROOT / 'pretrained',
)
display(pd.Series(trainer.describe(), name='Value').to_frame())
if HEADLESS:
    import shutil
    shutil.rmtree(INPUT_ROOT / 'pretrained', ignore_errors=True)
print('Document encoder:', trainer.model.document_encoder)
print('Residual classifier:', trainer.model.residual)

## 4. Execute one bounded slice
AdamW uses encoder/head learning rates of `1e-5`/`1e-4`, gradient clipping, and a scheduler spanning all epochs. Gradients accumulate over 16 articles by default. The session uses the configured checkpoint export reserve and pauses only at optimizer boundaries. Validation and final test evaluation can continue across slices too. Queueing and file transfers add elapsed time outside the execution budget; the deadline is soft at the granularity of a step/save.

In [ ]:
if ACTION == 'train':
    result = trainer.run(session_started_at=SESSION_STARTED)
elif ACTION == 'evaluate':
    result = trainer.evaluate_test(OUTPUT_ROOT, session_started_at=SESSION_STARTED)
else:
    raise ValueError(f'Unknown action: {ACTION}')
display(pd.Series({key: value for key, value in result.items() if key != 'history'}, name='Result').to_frame())

## 5. Results and portable checkpoint
Validation macro-F1 selects the development checkpoint. Read its selected epoch from `best_model/selection.json` before preparing a separate full-data payload. `model/` in a completed full-data archive is the deployment model; the article class definitions in `scripts/models.py` must be imported when loading it. Pooling attention is a diagnostic, not an annotated evidence span.

In [ ]:
display(pd.DataFrame(trainer.state['history']))
plot_history(trainer.state['history'], output_path=OUTPUT_ROOT / 'training_curves.png')
checkpoint = OUTPUT_ROOT / 'resume.zip'
print(f'Checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1e9:.2f} GB)')
print('Next session: upload this archive with the SAME immutable payload.')

# The CLI downloads archives, metrics, curves, and the executed notebook.
DOWNLOAD_CHECKPOINT = False
if DOWNLOAD_CHECKPOINT and not HEADLESS and Path('/content').exists():
    from google.colab import files
    files.download(str(checkpoint))